In [ ]:
JOBS_GPU0 = ['icarl_vna_b0__inc10__s1993']
JOBS_GPU1 = ['icarl_vna__inc10__s1993']
JOBS_CPU = []
RUN_TAG = 'vna_v3_pilot'
PRESET = 'Q2'
REPO_COMMIT = 'd461ce96816ce6ae7e1bbdde2c78d5bed7cc20ab'
SOURCE_DIGEST = None
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000


# VNA v3 decisive pilot
Attach source bundle and previous archives to resume. Q2/O1 produce shared authority schedules for Q1/Q3 and O2/O3. CPU variants do not require GPU or CIFAR-100. Preflight records calibration and continues training. C-Flat is deferred; Q3 GPU1 is empty.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vna_pilot_b3")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
if any(j.startswith(('er__','er_','erace')) for j in JOBS_GPU0+JOBS_GPU1):
    os.environ['VNA_FETCH_ONLINE']='1'
    subprocess.run([sys.executable,'-m','pip','install','-q','wandb','kornia','umap-learn','einops'],check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "diagnostics/signal_curve.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO), str(GROMO_SRC), str(REFERENCE)]


MU_REL_DEFAULT = 1e-2
OFFICIAL_MU = 1e-2
MU_SOURCE = "preregistered-pilot"


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs')
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from scripts.collect_parallel import validate_result
ROOT = STAGE_B / RUN_TAG
ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')

def completed(job):
    directory = ROOT/job
    path = directory/'summary.json'
    if not path.exists(): return False
    summary = json.loads(path.read_text())
    if summary.get('status') != 'COMPLETED': return False
    protocol = json.loads((directory/'protocol.json').read_text())
    tasks = json.loads((directory/'tasks.json').read_text())
    validate_result(job, summary, protocol, tasks, REPO_COMMIT, SOURCE_DIGEST)
    return True

def setting_for(job):
    if job.startswith('b5_'): return 'regression'
    if job.startswith('pm_'): return 'mnist'
    if job.startswith(('er__','er_','erace')): return 'online'
    return 'icarl'

def command(job):
    if job in ('b5_l2','b5_snp','b5_redo'):
        return [sys.executable,'-u','-m','experiments.competitors','b5','--arm',job[3:],'--out',str(ROOT/job),'--resume']
    label = job.split('__')[0]
    if label in ('finetune','joint','wa','podnet','bic','foster','der','memo','tagfex'):
        setting = 'b50inc10' if '__b50inc10' in job else 'inc10'
        return [sys.executable,'-u','-m','experiments.competitors','cil','--method',label,
                '--setting',setting,'--seed','1993','--out',str(ROOT/job),
                '--data-dir',str(DATA_ROOT),'--device','cuda:0','--resume']
    seed = re.search(r'__s(\d+)$',job).group(1)
    setting = setting_for(job)
    cmd = [sys.executable,'-u','-m','experiments.pilot_v3','--setting',setting,
           '--arm',label,'--seed',seed,'--out',str(ROOT/job),'--resume',
           '--device','cpu' if setting in ('regression','mnist') else 'cuda:0',
           '--workers','2']
    data_root = globals().get('MNIST_ROOT') if setting == 'mnist' else DATA_ROOT
    if data_root is not None: cmd += ['--data-dir',str(data_root)]
    if setting == 'online' and '__nf20__' in job: cmd += ['--online-nf','20']
    if label.startswith(('b5_cbp','pm_cbp')):
        rate = re.search(r'__rate([^_]+)',job)
        cmd += ['--replacement-rate',rate.group(1) if rate else '1e-3']
    if label not in ('b5_bp','b5_cbp','pm_bp','pm_cbp','icarl','er','erace'):
        schedule = ROOT/'_schedules'/f"{setting}{'_nf20' if setting == 'online' and '__nf20__' in job else ''}.json"
        if not schedule.exists():
            print(f'SKIP {job}: family schedule {schedule.name} missing; attach the authority archive and rerun', flush=True)
            return None
        cmd += ['--schedule',str(schedule)]
    return cmd

def prepare_schedule(setting, gpu=None):
    reduced = setting == 'online' and PRESET and PRESET.endswith('-R')
    filename = setting+('_nf20' if reduced else '')+'.json'
    destination = ROOT/'_schedules'/filename
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists(): return
    attached = []
    for path in Path('/kaggle/input').rglob(filename):
        try: value=json.loads(path.read_text())
        except (ValueError,OSError): continue
        if value.get('code_commit') == REPO_COMMIT and value.get('source_digest') == SOURCE_DIGEST and value.get('setting') == setting:
            attached.append((path,value))
    if attached:
        if any(value != attached[0][1] for _,value in attached): raise ValueError('conflicting authoritative schedules')
        shutil.copy2(attached[0][0],destination); return
    authority = {'icarl':'Q2','online':'O1'}
    authority_preset=authority.get(setting,'')+('-R' if reduced else '')
    if setting in authority and PRESET != authority_preset:
        # Baselines need no schedule; scheduled arms are skipped until the archive is attached.
        print(f'NOTE: attach {authority_preset} archive with {setting}.json to run scheduled {PRESET} arms', flush=True)
        return
    arm = dict(regression='b5_bp_vna_gir',mnist='pm_bp_vna',icarl='icarl_vna',online='er_vna')[setting]
    seed = '1993' if setting == 'icarl' else '0'
    cmd = [sys.executable,'-u','-m','experiments.pilot_v3','--setting',setting,'--arm',arm,
           '--seed',seed,'--out',str(ROOT/'_preflight'/setting),'--device','cpu' if gpu is None else 'cuda:0',
           '--resume','--preflight-only','--schedule-output',str(destination)]
    data_root = globals().get('MNIST_ROOT') if setting == 'mnist' else DATA_ROOT
    if data_root is not None: cmd += ['--data-dir',str(data_root)]
    if reduced: cmd += ['--online-nf','20']
    if gpu is None:
        from experiments.vna_resume import launch_logged
        return launch_logged(cmd,cwd=REPO,env={**ENV,'CUDA_VISIBLE_DEVICES':''},log_path=RUN_ROOT/f'{RUN_TAG}_{setting}_preflight.log',
            lane='CPU',job=setting+'_preflight',lanes=lane_state,paused=session_paused,deadline=VNA_DEADLINE)
    return launch(cmd,gpu,setting+'_preflight',RUN_ROOT/f'{RUN_TAG}_{setting}_preflight.log')

def run_gpu(gpu,jobs):
    for job in jobs:
        if pause_due(): break
        if completed(job): continue
        cmd = command(job)
        if cmd is None: continue
        if not launch(cmd,gpu,job,RUN_ROOT/f'{RUN_TAG}_{job}.log'): break
        archive_outputs()

def run_cpu(job):
    if pause_due() or completed(job): return
    cmd = command(job)
    if cmd is None: return
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd,cwd=REPO,env={**ENV,'CUDA_VISIBLE_DEVICES':''},
        log_path=RUN_ROOT/f'{RUN_TAG}_{job}.log',lane=job,job=job,lanes=lane_state,
        paused=session_paused,deadline=VNA_DEADLINE)

try:
    v3jobs = [j for j in JOBS_GPU0+JOBS_GPU1+JOBS_CPU if j not in ('b5_l2','b5_snp','b5_redo')
              and j.split('__')[0].startswith(('b5_','pm_','icarl','er'))]
    for setting in sorted({setting_for(j) for j in v3jobs}):
        prepare_schedule(setting,0 if setting in ('icarl','online') else None)
        if pause_due(): break
    archive_outputs()
    if not pause_due():
        with ThreadPoolExecutor(max_workers=2) as gpu_pool, ThreadPoolExecutor(max_workers=4) as cpu_pool:
            futures=[gpu_pool.submit(run_gpu,0,JOBS_GPU0),gpu_pool.submit(run_gpu,1,JOBS_GPU1)]
            futures += [cpu_pool.submit(run_cpu,job) for job in JOBS_CPU]
            for future in futures: future.result()
finally:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs()
from scripts.collect_parallel import collect, report
report(collect([ROOT.parent],RUN_TAG))
if session_paused.is_set(): print('PAUSED: attach output archive and rerun to resume')
if archive_errors: raise RuntimeError(archive_errors)
